# Test avec GradientBoostingRegressor

In [1]:
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from pycaret_optuna.modeling import build_pipeline, cross_val_r2, split_train_test

In [2]:
df = pd.read_csv("../data/gold.csv")

In [3]:
# column_titles = df.columns.tolist()
# column_titles

In [4]:
# Séparation entraînement / test (80 / 20, random_state=42)
X_train, X_test, y_train, y_test = split_train_test(df)

# Créer un objet GradientBoostingRegressor
# model = GradientBoostingRegressor(random_state=42, verbose=False)
# Meilleurs hyperparamètres trouvés par Optuna (validation croisée, voir plus bas)
model = GradientBoostingRegressor(
    n_estimators=1000,
    learning_rate=0.061919706101772935,
    max_depth=7,
    min_samples_split=10,
    subsample=0.7000000000000001,
    loss="squared_error",
    criterion="squared_error",
    max_features="log2",
    random_state=42,
    verbose=False,
)


# Créer un pipeline avec le préprocesseur et le modèle GradientBoostingRegressor
pipe = build_pipeline(model)

# Entraîner le pipeline sur les données d'entraînement
pipe.fit(X_train, y_train)
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

print("Mean squared error : ")
print("TRAIN :", mean_squared_error(y_train, y_pred_train))
print("TEST :", mean_squared_error(y_test, y_pred_test))

# print("/////////////////////////////////////////////////////")

print("Mean absolute error : ")
print("TRAIN :", mean_absolute_error(y_train, y_pred_train))
print("TEST :", mean_absolute_error(y_test, y_pred_test))

# print("/////////////////////////////////////////////////////")

print("R2 score : ")
print("TRAIN :", r2_score(y_train, y_pred_train))
print("TEST :", r2_score(y_test, y_pred_test))

Mean squared error : 
TRAIN : 1.6895824570742185e-05
TEST : 9.35662782120942
Mean absolute error : 
TRAIN : 0.003269477038507272
TEST : 1.8420577305536114
R2 score : 
TRAIN : 0.9999998055121151
TEST : 0.8724104032550198


In [5]:
# with mlflow.start_run():
#     mlflow.log_param("model", best_model.named_steps['model'].__class__.__name__)
#     mlflow.log_param("n_estimators", best_model.named_steps['model'].n_estimators)
#     mlflow.log_param("max_depth", best_model.named_steps['model'].max_depth)
#     mlflow.log_metric("mae", mae)
#     mlflow.log_metric("rmse", rmse)
#     mlflow.log_metric("r2", r2)
#     mlflow.sklearn.log_model(best_model, "model")

### Optuna (recherche des hyperparametres)

In [6]:
import os

import optuna

In [7]:
def objective(trial):
    # Définition des hyperparamètres à optimiser
    n_estimators = trial.suggest_int("n_estimators", 100, 1000, step=100)
    learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1, log=True)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    subsample = trial.suggest_float("subsample", 0.1, 1.0, step=0.1)
    loss = trial.suggest_categorical(
        "loss", ["absolute_error", "squared_error", "huber", "quantile"]
    )
    criterion = trial.suggest_categorical("criterion", ["squared_error", "friedman_mse"])
    max_features = trial.suggest_categorical("max_features", [1.0, "sqrt", "log2"])

    # Création du modèle avec les hyperparamètres suggérés
    model = GradientBoostingRegressor(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        subsample=subsample,
        loss=loss,
        criterion=criterion,
        max_features=max_features,
        random_state=42,
    )

    # Validation croisée du pipeline complet (préprocesseur + modèle) sur le jeu
    # d'entraînement uniquement : le jeu de test reste réservé à l'évaluation finale
    r2 = cross_val_r2(model, X_train, y_train)

    return r2

In [8]:
# Graine fixée pour que la recherche soit reproductible
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
# Nombre d'essais : 100 par défaut, réductible via la variable d'environnement N_TRIALS (CI)
n_trials = int(os.environ.get("N_TRIALS", 100))
study.optimize(objective, n_trials=n_trials)

[I 2026-10-09 08:29:07,007] A new study created in memory with name: no-name-0a50e593-f5f6-4cb1-a20a-77e283b76ede


[I 2026-10-09 08:29:11,082] Trial 0 finished with value: 0.8467407310329265 and parameters: {'n_estimators': 400, 'learning_rate': 0.07969454818643935, 'max_depth': 8, 'min_samples_split': 7, 'subsample': 0.2, 'loss': 'huber', 'criterion': 'squared_error', 'max_features': 1.0}. Best is trial 0 with value: 0.8467407310329265.


[I 2026-10-09 08:29:11,656] Trial 1 finished with value: 0.47806533964004866 and parameters: {'n_estimators': 200, 'learning_rate': 0.002327067708383781, 'max_depth': 5, 'min_samples_split': 6, 'subsample': 0.5, 'loss': 'squared_error', 'criterion': 'friedman_mse', 'max_features': 1.0}. Best is trial 0 with value: 0.8467407310329265.


[I 2026-10-09 08:29:12,332] Trial 2 finished with value: 0.5089025505435835 and parameters: {'n_estimators': 600, 'learning_rate': 0.001238513729886093, 'max_depth': 7, 'min_samples_split': 3, 'subsample': 0.1, 'loss': 'squared_error', 'criterion': 'friedman_mse', 'max_features': 'log2'}. Best is trial 0 with value: 0.8467407310329265.


[I 2026-10-09 08:29:12,788] Trial 3 finished with value: 0.6900620703920239 and parameters: {'n_estimators': 100, 'learning_rate': 0.06586289317583113, 'max_depth': 5, 'min_samples_split': 7, 'subsample': 0.4, 'loss': 'quantile', 'criterion': 'friedman_mse', 'max_features': 'log2'}. Best is trial 0 with value: 0.8467407310329265.


...

[I 2026-10-09 08:31:25,791] Trial 95 finished with value: 0.8759492091009695 and parameters: {'n_estimators': 1000, 'learning_rate': 0.05853977765688106, 'max_depth': 7, 'min_samples_split': 9, 'subsample': 0.8, 'loss': 'squared_error', 'criterion': 'squared_error', 'max_features': 'log2'}. Best is trial 51 with value: 0.8829390407673168.


[I 2026-10-09 08:31:27,352] Trial 96 finished with value: 0.8696145710750317 and parameters: {'n_estimators': 1000, 'learning_rate': 0.0707923371914844, 'max_depth': 7, 'min_samples_split': 10, 'subsample': 0.6, 'loss': 'squared_error', 'criterion': 'squared_error', 'max_features': 'log2'}. Best is trial 51 with value: 0.8829390407673168.


[I 2026-10-09 08:31:28,790] Trial 97 finished with value: 0.8808565405175335 and parameters: {'n_estimators': 900, 'learning_rate': 0.06489403480057071, 'max_depth': 7, 'min_samples_split': 10, 'subsample': 0.7000000000000001, 'loss': 'squared_error', 'criterion': 'squared_error', 'max_features': 'log2'}. Best is trial 51 with value: 0.8829390407673168.


[I 2026-10-09 08:31:30,170] Trial 98 finished with value: 0.875651964208035 and parameters: {'n_estimators': 900, 'learning_rate': 0.03574026467420252, 'max_depth': 6, 'min_samples_split': 10, 'subsample': 0.6, 'loss': 'squared_error', 'criterion': 'squared_error', 'max_features': 'log2'}. Best is trial 51 with value: 0.8829390407673168.


[I 2026-10-09 08:31:31,810] Trial 99 finished with value: 0.8865011487368758 and parameters: {'n_estimators': 1000, 'learning_rate': 0.061919706101772935, 'max_depth': 7, 'min_samples_split': 10, 'subsample': 0.7000000000000001, 'loss': 'squared_error', 'criterion': 'squared_error', 'max_features': 'log2'}. Best is trial 99 with value: 0.8865011487368758.


In [9]:
best_params = study.best_params
best_value = study.best_value

In [10]:
print(best_params)
print(best_value)

{'n_estimators': 1000, 'learning_rate': 0.061919706101772935, 'max_depth': 7, 'min_samples_split': 10, 'subsample': 0.7000000000000001, 'loss': 'squared_error', 'criterion': 'squared_error', 'max_features': 'log2'}
0.8865011487368758


### Évaluation finale sur le jeu de test

In [11]:
# Pipeline avec les meilleurs hyperparamètres trouvés par Optuna,
# entraîné sur le jeu d'entraînement puis évalué une seule fois sur le jeu de test
best_pipe = build_pipeline(GradientBoostingRegressor(**best_params, random_state=42))
best_pipe.fit(X_train, y_train)
y_pred_test = best_pipe.predict(X_test)

print("TEST MSE :", mean_squared_error(y_test, y_pred_test))
print("TEST MAE :", mean_absolute_error(y_test, y_pred_test))
print("TEST R2 :", r2_score(y_test, y_pred_test))

TEST MSE : 9.35662782120942
TEST MAE : 1.8420577305536114
TEST R2 : 0.8724104032550198
